# 🐳 PHASE 2: Docker, Netzwerk & Reverse Proxy (Traefik + OAuth2-Proxy)
**Projekt: Mai_AI (MaiOmni) — Das generative, private KI-Betriebssystem**

In dieser Phase schalten wir das schützende **Traefik-Gateway** und den **Google OAuth2-Proxy** vor dein System. Dieses Notebook dient als zentrale **Steuerungseinheit für die Infrastruktur-Härtung** und setzt die Sicherheitsarchitektur gemäß der **Matrix der Infrastruktur-Vollständigkeit** in 5 modularen Schritten um.

⚡ **Zentralisierte Konfigurations-Architektur (`config/docker_global.json`):**
Alle globalen Einstellungen, Ports, Volume-Namen, Quotas, Security-Flags (`read_only`, `cap_drop`, `user`) und Lifecycle-Timeouts werden **ausschließlich in der zentralen Konfigurationsdatei `config/docker_global.json`** gepflegt. Sämtliche Skripte in `src/docker_py/` beziehen ihre Werte dynamisch zur Laufzeit aus dieser Konfiguration – es existieren keinerlei statische Hardcoding-Werte mehr im Code.

---
### 🎯 Die 5 Kern-Schritte dieser Phase:
1. **Schritt 1:** Docker Daemon & Host-Integrität (Hardening-Check & Selbstheilung)
2. **Schritt 2:** Infrastruktur-Netzwerk & Datenschnittstellen (`mai-ai_network`, Named Volumes & Gateway-Audit)
3. **Schritt 3:** Dynamische Benutzer-Isolation & Storage-Quotas (`/data/users/<user_id>/`, Header-Routing `X-Forwarded-User`)
4. **Schritt 4:** Asynchrone SQLite-Inbox für Chat-Requests (`HTML-Interface -> Request -> Chat-Antwort` Entkopplung)
5. **Schritt 5:** Container Hardening & Automated Idle-Lifecycle (`Read-Only Dateisystem`, `Cap-Drop: [ALL]`, `Non-Root`, `Ressourcen-Limits` & 10-Minuten Idle-Timeout)

---

### 🛠️ Schritt 1: Docker Daemon & Host-Integrität (Hardening-Check)
Wir laden zuerst die globale Konfiguration aus `config/docker_global.json` und prüfen mittels Python Docker SDK, ob die Docker Engine läuft und erreichbar ist. Falls der Daemon offline ist, startet die integrierte Selbstheilungslogik den Dienst plattformspezifisch mit den konfigurierten Timeouts und Retry-Zyklen.

Gleichzeitig führt dieser Schritt einen umfassenden **Hardening-Vorab-Check** durch:
*   **Daemon-Integrität:** Überprüfung des sicheren Docker-Sockets und der Engine-Version.
*   **Hardening-Fähigkeiten:** Validierung der Host-Unterstützung für ein **Read-Only Dateisystem**, **Cap-Drop [ALL]**, **Non-Root**-Ausführung sowie harte **Ressourcen-Limits** (Cgroup v2).
*   **Ressourcen-Audit:** Erfassung von Host-CPUs und verfügbarem Arbeitsspeicher als Basis für nachfolgende Kapseln.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.docker_py import (
    load_docker_global_config,
    run_step1_daemon_check
)

# 1. Zentrale Konfiguration aus config/docker_global.json dynamisch laden
docker_config = load_docker_global_config()

# 2. Schritt 1: Docker Daemon Verbindung & Hardening-Fähigkeiten prüfen
success, client, daemon_meta = run_step1_daemon_check(config=docker_config)

[!] Docker-Daemon ist aktuell nicht erreichbar (Versuch 1/3).
[...] Starte Docker-Umgebung für Plattform 'win32'... Bitte warten.
[...] Warte 6 Sekunden auf Initialisierung...
[✓] Schritt 1 erfolgreich konfiguriert: Docker-Daemon ist aktiv (Server Version: 29.7.2)
    -> Hardening-Fähigkeiten bestätigt: Read-Only Dateisystem, Cap-Drop [ALL], Non-Root & Ressourcen-Limits.
    -> Host-Ressourcen: 8 CPUs | 7.68 GB RAM | Cgroup: 2


### 🌐 Schritt 2: Infrastruktur-Netzwerk & Datenschnittstellen initialisieren
Die Benutzer-Container kommunizieren niemals unkontrolliert über das Host-Netzwerk, sondern ausschließlich über das dedizierte Brücken-Netzwerk `mai-ai_network` mit dem Traefik-Gateway. Alle Netzwerknamen, Named Volumes und `.env`-Schlüssel werden dynamisch aus `config/docker_global.json` bezogen.

*   **Netzwerk-Isolation:** Das `mai-ai_network` kapselt sämtliche internen Container. Direkte Kommunikation zwischen einzelnen Benutzer-Kapseln ist unterbunden.
*   **Persistente Datenschnittstellen:** Initialisierung der geschützten Named Volumes (`mai_ai_local_models`, `mai_ai_db_data`, `mai_ai_config`) zur Datenhaltung der KI-Engine und des Authentifizierungs-Gateways.
*   **Gateway-Preflight-Audit:** Validierung der `.env`-Konfiguration (Google Client-ID, Domain-Parameter, Cookie-Secret) vor dem Start des Traefik/OAuth2-Stacks.

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.docker_py import run_step2_infrastructure_setup

# Schritt 2: Netzwerk, Volumes & Ingress-Gateway Schnittstellen initialisieren
success, infra_meta = run_step2_infrastructure_setup(
    client=client if 'client' in locals() else None,
    config=docker_config if 'docker_config' in locals() else None
)

[✓] Schritt 2 erfolgreich konfiguriert: Infrastruktur-Netzwerk & Datenschnittstellen aktiv
    -> Brücken-Netzwerk: 'mai-ai_network' (existing)
    -> Datenschnittstelle: 'mai_ai_local_models' (existing)
    -> Datenschnittstelle: 'mai_ai_db_data' (existing)
    -> Datenschnittstelle: 'mai_ai_config' (existing)
    -> Hinweis Gateway-Konfiguration: '.env' noch nicht angelegt (wird bei Start_AI.py initialisiert)
    -> KI-Engine Container Status: 'mai_ai_ollama_engine' -> not_found


### 🔒 Schritt 3: Dynamische Benutzer-Isolation & Storage-Quotas
Um Datenvermischungen und unbefugte Zugriffe technisch auszuschließen, erhält jeder Benutzer eine streng isolierte Umgebung (**Dynamische Benutzer-Isolation**).

*   **User-Space-Verzeichnisse:** Automatische Erstellung getrennter Verzeichnisse unter `/data/users/<user_id>/` (`workspace`, `history`, `inbox`, `output`).
*   **Isoliertes Mount-Mapping:** Der Benutzer-Workspace wird im Container als `rw` gemountet, während zentrale Sicherheitsregeln als **Read-Only Dateisystem** (`ro`) eingebunden werden.
*   **Storage-Quotas & Ressourcen-Limits:** Kontinuierliche Überwachung des Festplattenverbrauchs (definiert in `config/docker_global.json`, z.B. 500 MB), um Ressourcen-Erschöpfung zu verhindern.
*   **Traefik-Header-Routing:** Dynamische Erstellung von Routing-Regeln basierend auf dem `X-Forwarded-User`-Header des Google OAuth2-Proxys.

In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.docker_py import run_step3_user_isolation

# Schritt 3: Dynamische Benutzer-Isolation für Test-User einrichten & Quotas prüfen
success, isolation_meta = run_step3_user_isolation(
    user_id="user_alice@mai-ai.local",
    config=docker_config if 'docker_config' in locals() else None
)

[✓] Schritt 3 erfolgreich konfiguriert: Dynamische Benutzer-Isolation eingerichtet
    -> User-Space Pfad: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\data/users\user_alice_mai-ai_local
    -> Storage-Quota: 0.0 MB / 500.0 MB (0.0% belegt - Status: OK)
    -> Header-Routing: X-Forwarded-User == 'user_alice@mai-ai.local'
    -> Isolierte Mounts: /workspace (rw), /app/security (ro)


### 📬 Schritt 4: Asynchrone SQLite-Inbox für Chat-Requests
Die Kommunikation zwischen dem Web-Frontend und der KI-Rechenengine wird über eine transiente **SQLite-Inbox** entkoppelt. Alle Datenbanknamen und Standardmodelle werden direkt aus `config/docker_global.json` geladen.

*   **Pipeline:** `HTML-Interface -> Request (SQLite-Inbox) -> Dispatcher/Engine -> Chat-Antwort`.
*   **Asynchrones Queuing:** Anfragen werden in der Tabelle `incoming_requests` mit dem Status `PENDING` gepuffert.
*   **Antwort-Zustellung:** Sobald die Engine die Antwort berechnet hat, wird sie in `chat_responses` abgelegt und der Request auf `COMPLETED` gesetzt.
*   **Audit-Protokollierung:** Sämtliche Transaktionen werden im isolierten User-Space nachvollziehbar protokolliert.

In [4]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.docker_py import run_step4_sqlite_inbox_test

# Schritt 4: SQLite-Inbox initialisieren und vollständigen Request-Response-Zyklus testen
success, inbox_meta = run_step4_sqlite_inbox_test(
    user_id="user_alice@mai-ai.local",
    sample_prompt="Hallo Mai_AI, wie ist mein System-Status?",
    config=docker_config if 'docker_config' in locals() else None
)

[✓] Schritt 4 erfolgreich konfiguriert: SQLite-Inbox für asynchrones Messaging aktiv
    -> Inbox-Datenbank: c:\Users\MacBookAir\Desktop\GitHub\Offline_AI\data/users\user_alice_mai-ai_local\inbox\user_inbox.db
    -> Request Enqueue (ID: 1): "Hallo Mai_AI, wie ist mein System-Status?" (Status: PENDING -> COMPLETED)
    -> Chat-Antwort verarbeitet (ID: 1, Modell: 'Codestral'): "System aktiv: Deine isolierte Sandbox läuft geschützt im mai-ai_network."
    -> Nachrichten-Pipeline: HTML-Interface -> Request -> Chat-Antwort erfolgreich verifiziert.


### 🛡️ Schritt 5: Container Hardening & Automated Idle-Lifecycle
In diesem Schritt werden die maximalen Sicherheitsrichtlinien für alle Benutzer-Kapseln scharfgeschaltet und der automatisierte Lebenszyklus aktiviert – alle Parameter stammen aus `config/docker_global.json`:

*   **Read-Only Dateisystem (`read_only: True`):** Das Root-Dateisystem des Containers ist unveränderlich geschützt; schreibbar sind nur gemountete User-Spaces und flüchtige `tmpfs`-Pfade.
*   **Cap-Drop Minimalrechte (`cap_drop: [ALL]`):** Sämtliche Linux-Kernel-Privilegien (Root-Rechte, Raw Sockets, Admin-Caps) werden dem Container entzogen.
*   **Non-Root Ausführung (`user: 1000:1000`):** Prozesse laufen zwingend als unprivilegierter Benutzer.
*   **Harte Ressourcen-Limits:** Begrenzung jeder Kapsel auf maximal 1 GB RAM (`mem_limit="1g"`) und 1 CPU-Kern (`nano_cpus=1000000000`), um Denial-of-Service (DoS) abzuwehren.
*   **Automated Idle-Lifecycle:** Überschreitet die Inaktivität einer Chat-Sitzung 10 Minuten (definiert in `config/docker_global.json`), fährt der Container (`container.stop()`) eigenständig herunter. Bei Eintreffen einer neuen Anfrage in der **SQLite-Inbox** erfolgt eine sofortige On-Demand-Reaktivierung (Wakeup).

In [5]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.docker_py import run_step5_hardening_and_lifecycle_check

# Schritt 5: Container-Hardening Spezifikation & Automated Idle-Lifecycle validieren
success, hardening_meta = run_step5_hardening_and_lifecycle_check(
    user_id="user_alice@mai-ai.local",
    config=docker_config if 'docker_config' in locals() else None
)

[✓] Schritt 5 erfolgreich konfiguriert: Container Hardening & Automated Idle-Lifecycle aktiv
    -> Read-Only Dateisystem: True (Root-Dateisystem ist schreibgeschützt)
    -> Kernel Cap-Drop: ['ALL'] (Sämtliche Root-Capabilities entzogen)
    -> Non-Root Ausführung: user='1000:1000' (UID/GID 1000 ohne Root-Rechte)
    -> Ressourcen-Limits: RAM=1g | CPU=1.0 Core(s)
    -> Automated Idle-Lifecycle: Inaktivität >10 Min (660s) -> Auto-Shutdown: True
    -> On-Demand Reaktivierung: Bei neuem Request -> Sofortiges Wakeup / Re-Launch


---
### 📋 6. Matrix der Infrastruktur-Vollständigkeit (Mai_AI System-Audit)

Alle 5 Schritte wurden erfolgreich konfiguriert und architektonisch an die zentrale `config/docker_global.json` gekoppelt:

| Funktionsbereich | Konfiguration (`config/docker_global.json`) | Status |
| :--- | :--- | :--- |
| **Schritt 1: Docker Daemon & Host-Integrität** | Socket-Verbindung, Cgroup v2, Hardening-Audit | **[✓] Aktiv** |
| **Schritt 2: Netzwerk & Datenschnittstellen** | `mai-ai_network`, Named Volumes, Ingress-Audit | **[✓] Aktiv** |
| **Schritt 3: Dynamische Benutzer-Isolation** | `/data/users/<user_id>/`, Storage-Quotas (500 MB), Traefik-Header | **[✓] Aktiv** |
| **Schritt 4: SQLite-Inbox Messaging** | Entkoppelte Queue `HTML -> Request -> Chat-Antwort` (`user_inbox.db`) | **[✓] Aktiv** |
| **Schritt 5: Container Hardening & Idle-Lifecycle** | `read_only: True`, `cap_drop: [ALL]`, `Non-Root`, 10-Min Auto-Shutdown | **[✓] Aktiv** |

---
### 🔄 Was kommt als Nächstes? (Ausblick & Roadmap)
Die gesamte Docker-Infrastruktur, das Ingress-Routing, die Benutzer-Isolation und die gehärteten Sicherheits-Kapseln sind nun einsatzbereit und zentral steuerbar.

Fahre nun mit dem nächsten Knotenpunkt fort, um das Streamlit-Nutzer-Image und das HTML-Iframe-Embedding in Betrieb zu nehmen:
👉 **[03_html_embed.ipynb](file:notebooks/03_html_embed.ipynb)**